In [0]:
dbutils.widgets.text("catalog", "")
spark.sql(f"USE CATALOG `{dbutils.widgets.get('catalog')}`")
spark.sql("USE SCHEMA default")

print(dbutils.widgets.get("catalog"))

In [0]:
# %sql
# CREATE WIDGET TEXT catalog DEFAULT '';
# USE CATALOG IDENTIFIER(:catalog);
# USE SCHEMA default;

In [0]:
dbutils.widgets.text("job_run_id", "")

print(dbutils.widgets.get("job_run_id"))

In [0]:
from pyspark.sql.functions import current_timestamp, current_user, lit, input_file_name

from pyspark.sql.types import StructType, StructField, TimestampType, StringType

# Create table if not exists, add job_id column
spark.sql("""
CREATE TABLE IF NOT EXISTS test_events (
  event_time TIMESTAMP,
  message STRING,
  runtime_user STRING,
  job_id STRING
)
""")

job_id = dbutils.widgets.get("job_run_id")
if not job_id:
    job_id = "not run as a job"

schema = StructType([
    StructField("event_time", TimestampType(), True),
    StructField("message", StringType(), True),
    StructField("runtime_user", StringType(), True),
    StructField("job_id", StringType(), True)
])

df = spark.createDataFrame(
    [(None, "Hello world", None, job_id)],
    schema=schema
).withColumn("event_time", current_timestamp()
).withColumn("runtime_user", current_user())

df.write.mode("append").option("mergeSchema", "true").insertInto("test_events")

display(spark.table("test_events").orderBy("event_time", ascending=False))